In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7" # Check using nvidia-smi in terminal and choose GPUs that are not being used
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from haversine import haversine
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Detect device
device_type = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device_type.upper()} for training")

torch.manual_seed(42)
np.random.seed(42)

Using CUDA for training


In [3]:
TRAIN_CSV_PATH = "osv-5m_subset/train-150k.csv"
TEST_CSV_PATH  = "osv-5m_subset/test-10k.csv"
TRAIN_IMG_DIR  = "osv-5m_subset/train-150k"
TEST_IMG_DIR   = "osv-5m_subset/test-10k"

if not os.path.exists(TRAIN_CSV_PATH):
    raise FileNotFoundError(f"CSV file not found: {TRAIN_CSV_PATH}")
if not os.path.exists(TEST_CSV_PATH):
    raise FileNotFoundError(f"CSV file not found: {TEST_CSV_PATH}")
if not os.path.exists(TRAIN_IMG_DIR):
    raise FileNotFoundError(f"Image folder not found: {TRAIN_IMG_DIR}")
if not os.path.exists(TEST_IMG_DIR):
    raise FileNotFoundError(f"Image folder not found: {TEST_IMG_DIR}")

train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

In [4]:
class OSV5MDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['id']}.jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        target = torch.tensor([row['latitude'], row['longitude']], dtype=torch.float32)
        return image, target

# Transform to 224x224 for ResNet tranfer learning
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [5]:
class OSV5MModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, train_image_dir, test_image_dir, batch_size=32):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.train_image_dir = train_image_dir
        self.test_image_dir = test_image_dir
        self.batch_size = batch_size

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_dataset = OSV5MDataset(self.train_df, self.train_image_dir, self.transform)
        self.test_dataset = OSV5MDataset(self.test_df, self.test_image_dir, self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

In [ ]:
import torch.nn.functional as F

def haversine_loss(pred, target):
    """
    Differentiable Haversine distance loss (in km).
    pred, target: tensors of shape [batch_size, 2] (lat, lon)
    """
    # Convert degrees to radians
    lat1, lon1 = torch.deg2rad(pred[:, 0]), torch.deg2rad(pred[:, 1])
    lat2, lon2 = torch.deg2rad(target[:, 0]), torch.deg2rad(target[:, 1])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    
    a = torch.clamp(a, 1e-7, 1.0 - 1e-7)

    c = 2 * torch.arcsin(torch.sqrt(a))
    km = 6371 * c  # Earth's radius in kilometers

    # Check for NaN
    if torch.isnan(km).any() or torch.isinf(km).any():
        return torch.tensor(0.0, device=km.device, requires_grad=True)
    
    return km.mean()


class ResNetRegressorPLM(pl.LightningModule):
    def __init__(self, learning_rate=1e-4, backbone="resnet18", loss_type="hybrid", lambda_hav=0.001, freeze_early=True):
        """
        Args:
            learning_rate (float): optimizer learning rate
            backbone (str): 'resnet18' or 'resnet34'
            loss_type (str): 'mse', 'haversine', or 'hybrid'
            lambda_hav (float): weight for haversine loss when using hybrid
        """
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.loss_type = loss_type
        self.lambda_hav = lambda_hav

        # Choose model backbone
        if backbone == "resnet18":
            self.model = models.resnet18(weights="IMAGENET1K_V1")
        elif backbone == "resnet34":
            self.model = models.resnet34(weights="IMAGENET1K_V1")
        elif backbone == "resnet50":
            self.model = models.resnet50(weights="IMAGENET1K_V1")
        elif backbone == "resnet101":
            self.model = models.resnet101(weights="IMAGENET1K_V1")
        else:
            raise ValueError("Unsupported backbone")
        
        # Freeze early layers to preserve "big picture" of model
        if freeze_early:
            for name, param in self.model.named_parameters():
                if 'layer4' not in name and 'fc' not in name:
                    param.requires_grad = False

        # Replace final FC layer with deeper regression head
        self.model.fc = nn.Sequential(
            nn.Linear(self.model.fc.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),
            nn.Tanh()
        )

        for module in self.model.fc.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

        # Define basic MSE for comparison
        self.mse = nn.MSELoss()

    def forward(self, x):
        output = self.model(x)
        lat = output[:, 0] * 90
        lon = output[:, 1] * 180
        return torch.stack([lat, lon], dim=1)

    def compute_loss(self, outputs, targets):
        """Select and compute loss based on loss_type."""
        mse_loss = self.mse(outputs, targets)
        if self.loss_type == "mse":
            return mse_loss
        elif self.loss_type == "haversine":
            return haversine_loss(outputs, targets)
        elif self.loss_type == "hybrid":
            hav_loss = haversine_loss(outputs, targets)
            return mse_loss + self.lambda_hav * hav_loss
        else:
            raise ValueError(f"Unsupported loss type: {self.loss_type}")

    def training_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('train_loss', loss)
        self.log('train_rmse', rmse)
        return loss

    def test_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('test_loss', loss)
        self.log('test_rmse', rmse)

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.learning_rate)

In [ ]:
geo_module = OSV5MModule(
    train_df,
    test_df,
    TRAIN_IMG_DIR,
    TEST_IMG_DIR,
    batch_size=32
)

geo_model = ResNetRegressorPLM(
    learning_rate=1e-6,
    backbone="resnet101",
    loss_type="haversine",
    freeze_early=True
)

trainer = pl.Trainer(
    max_epochs=25,
    accelerator=device_type,
    devices=1,
    log_every_n_steps=10,
    gradient_clip_val=1.0
)

time_start = time.time()
trainer.fit(geo_model, datamodule=geo_module)
time_stop = time.time()

print(f"Training completed in {round(time_stop - time_start, 1)} seconds.")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4,5,6,7]

  | Name  | Type    | Params | Mode 
------------------------------------------
0 | model | ResNet  | 43.7 M | train
1 | mse   | MSELoss | 0      | train
------------------------------------------
16.1 M    Trainable params
27.5 M    Non-trainable params
43.7 M    Total params
174.724   Total estimated model params size (MB)
295       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=25` reached.


Training completed in 9535.3 seconds.


In [11]:
# Save the trained model weights
torch.save(geo_model.model.state_dict(), 'resnet101_geolocation_weights.pth')

# Or save the entire Lightning module (recommended)
trainer.save_checkpoint('resnet101_geolocation_full.ckpt')

print("Model saved successfully!")

Model saved successfully!


In [ ]:
# Use this cell only if loading
loaded_model = ResNetRegressorPLM(
    learning_rate=1e-5,
    backbone="resnet101",
    loss_type="haversine"
)
loaded_model.model.load_state_dict(torch.load('resnet101_geolocation_weights.pth'))
# loaded_model.eval()

geo_model = loaded_model

trainer = pl.Trainer(
    accelerator=device_type,
    devices=1,
    logger=False
)

geo_module = OSV5MModule(
    train_df,
    test_df,
    TRAIN_IMG_DIR,
    TEST_IMG_DIR,
    batch_size=32
)

In [ ]:
trainer.test(geo_model, datamodule=geo_module)

geo_model.eval()
preds, trues = [], []

for images, targets in geo_module.test_dataloader():
    with torch.no_grad():
        outputs = geo_model(images)
    preds.extend(outputs.cpu().numpy())
    trues.extend(targets.cpu().numpy())

# Convert to numpy arrays
preds = np.array(preds)
trues = np.array(trues)

# Clamp to valid geographic ranges
preds[:, 0] = np.clip(preds[:, 0], -90, 90)     # latitude
preds[:, 1] = np.clip(preds[:, 1], -180, 180)   # longitude
trues[:, 0] = np.clip(trues[:, 0], -90, 90)
trues[:, 1] = np.clip(trues[:, 1], -180, 180)

# Compute mean Haversine distance safely
errors = [haversine((trues[i][0], trues[i][1]), (preds[i][0], preds[i][1]))
          for i in range(len(trues))]
mean_error = np.mean(errors)

print(f"Mean Haversine error: {mean_error:.2f} km")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4,5,6,7]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss                   nan
        test_rmse                   nan
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
